# Fase 13 — Run ulang GRID-FIX di steel-pipe: P1-NoProj, 5 varian penempatan, 2 probe resep

**Tujuan.** Mengganti semua "angka sementara" di naskah (Tabel 3 varian parsial, Tabel 5 P1-NoProj, §4.3 + catatan
kaki ³ §5.6) dengan run yang memakai `deform_ss2d.py` **GRID-FIX** (sha256 `675445871a2e4d55…d9aa`) — versi yang sama
dengan P1 penuh steel-pipe (5 seed) dan seluruh run SWRD v1. B2 tidak memakai modul ini, jadi **B2 tidak diulang**.

- **Dataset:** steel-pipe (Yang et al. 2021) — Kaggle `benyxu/dsf-yolo-steel-tube-dataset`, `yolo/images/train2021`
  (2.726) + `val2021` (682), 8 kelas, 6.783 objek (1.381 di val). Nama kelas di YAML = nama lama (agar CSV sejalan
  dgn CSV lama); pemetaan ke naskah: `dense-gas-hole`→hollow-bead, `non-fusion`→overlap, `unfused-hole`→unfused.
- **Resep terkunci (identik semua run utama):** SGD, lr0=0.002, cos_lr=False, 150 epoch kontinu, patience=0,
  cache=True, amp=True, batch 8, imgsz 640, dari nol, TF32 mati.
- **Lingkungan dipin:** torch 2.11.0+cu128, GPU A100 (sama dgn P1 grid-fix & SWRD v1). Commit repo dicatat (Sel 4a).

### Daftar run (32) — jalankan berurutan sesuai prioritas

| Prioritas | MODEL | RUN_MODE | Seed | Run | Mengganti di naskah |
|---|---|---|---|---|---|
| 1 | `P1-NoProj` | `utama` | 1–5 | 5 | Tabel 5, §5.4 (i)–(ii), kontribusi 3, Abstrak/Intro "1 of 5" |
| 2 | `BackboneOnly`, `NeckOnly`, `BB-Late`, `BB-Early`, `BB-Late-Alt` | `utama` | 1–5 | 25 | Tabel 3, §5.2, interaksi §6.1, §8 |
| 3 | `P1` | `probe_adamw` | 1 | 1 | §4.3 & catatan kaki ³ (divergen di AdamW-auto) |
| 3 | `P1` | `probe_lr0.01` | 1 | 1 | §4.3 (divergen di SGD lr0 = 0.01) |

Progres (centang tiap selesai Sel 9):
- [ ] P1-NoProj s1 · [ ] s2 · [ ] s3 · [ ] s4 · [ ] s5
- [ ] BackboneOnly s1 · [ ] s2 · [ ] s3 · [ ] s4 · [ ] s5
- [ ] NeckOnly s1 · [ ] s2 · [ ] s3 · [ ] s4 · [ ] s5
- [ ] BB-Late s1 · [ ] s2 · [ ] s3 · [ ] s4 · [ ] s5
- [ ] BB-Early s1 · [ ] s2 · [ ] s3 · [ ] s4 · [ ] s5
- [ ] BB-Late-Alt s1 · [ ] s2 · [ ] s3 · [ ] s4 · [ ] s5
- [ ] probe_adamw · [ ] probe_lr0.01

**Estimasi waktu (kasar):** B2 ±2,7 jam/run di A100 (tracker); run deformable lebih lama → perkirakan 3–4,5 jam/run,
total ±100–140 jam A100. Prioritas 1 (5 run) cukup untuk menentukan nasib kontribusi 3.

### ATURAN — dikunci SEBELUM run pertama (jangan diubah setelah melihat hasil)

1. **Kolaps** = kriteria §4.4 naskah: loss latih pernah NaN/inf, ATAU `train/cls_loss` epoch 140 ≥ **1.5×** minimumnya
   di epoch 1–140. Dihitung otomatis di **Sel 8b** → `status_latih_gridfix.csv`.
2. **Run kolaps TIDAK diulang** dan tetap dilaporkan (mengulang = memilih hasil; `grid_sample` non-deterministik).
3. **Run terputus** (Colab putus/kuota habis) BUKAN run baru: jalankan ulang notebook dgn MODEL/SEED sama → Sel 6
   mendeteksi `BELUM_SELESAI` → resume di folder yang sama.
4. **Probe resep:** "divergen" = kolaps menurut kriteria no. 1. Bila probe **tidak** kolaps, kalimat §4.3/catatan kaki ³
   **ditulis ulang** sesuai hasil — probe TIDAK diulang untuk "mencari" divergensi.
5. Semua hasil dipakai **apa adanya**, termasuk bila mengubah narasi (mis. P1-NoProj ternyata stabil, atau satu
   varian parsial kolaps).

**Pengaman biaya:** (a) GPU selain A100 ditolak di awal Sel 2, sebelum unduhan (bisa dibuka dgn `ALLOW_NON_A100 = True`);
(b) bila ada sel gagal saat Run all, runtime dilepas otomatis setelah 60 detik (dipasang di awal Sel 1; tekan Stop di jeda
itu untuk menahan); (c) sel terakhir melepas runtime setelah semua selesai.

**Kompatibilitas numpy:** Ultralytics 8.2.29 memakai `np.trapz`, yang DIHAPUS mulai numpy 2.4. Patch 6.1 (Sel 4b) memakai
`np.trapezoid` (fungsi yang sama, angka identik) agar run tidak crash bila Colab memperbarui numpy.

**Keamanan folder:** semua run bernama `gf_…` → tidak bertabrakan dgn folder lama (Sel 6 tidak akan me-resume / melewati
checkpoint lama). `exist_ok=True` → Ultralytics tidak membuat folder `…2` (kasus seed 4 dulu).


## 0. Pin Lingkungan — torch 2.11.0+cu128 (WAJIB paling awal, sebelum `import torch`)

Colab kini memasang torch **2.11.0+cu130** (CUDA 13). Kernel `selective_scan` gagal di-compile di CUDA 13, dan
run P1-fix/Fase 10–12 sebelumnya memakai **2.11.0+cu128**. Sel ini memasang build cu128 (versi torch SAMA, beda
build CUDA) lalu me-RESTART runtime otomatis. **Setelah restart: Runtime → Run all SEKALI LAGI** — kali kedua sel ini
hanya mencetak "OK" dan notebook lanjut, memakai wheel `selective_scan` cu128 yang sudah ada di cache Drive.


In [ ]:
import importlib.metadata as _md, subprocess, sys, os
TORCH_PIN = "2.11.0+cu128"          # = lingkungan run P1-fix / Fase 10-12 (tracker: torch 2.11.0+cu128, py3.13)
_tv = _md.version("torch")
print("torch terpasang:", _tv)
if _tv != TORCH_PIN:
    assert "torch" not in sys.modules, "torch sudah ter-import di sesi ini -- Runtime > Restart session, lalu Run all"
    _vis = "0.26.0"   # pasangan RESMI torch 2.11.0 (metadata wheel: Requires-Dist torch==2.11.0) -- TIDAK
                      # diambil dari torchvision bawaan Colab (bisa sudah versi lain bila Colab naik ke torch 2.12+)
    _pkgs = [f"torch=={TORCH_PIN}", f"torchvision=={_vis}+cu128"]
    print("Memasang", _pkgs, "dari indeks PyTorch cu128 (sekali per sesi, beberapa menit)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pkgs,
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    print("Terpasang:", subprocess.run([sys.executable, "-c", "import torch,torchvision;print(torch.__version__, torch.version.cuda, torchvision.__version__)"],
                                       capture_output=True, text=True).stdout.strip())
    print("\n>>> Runtime akan RESTART sekarang. Setelah tersambung lagi: Runtime > Run all SEKALI LAGI. <<<")
    sys.stdout.flush()
    os.kill(os.getpid(), 9)
else:
    print(f"OK: torch = {TORCH_PIN} -- lanjut.")


## 1. Pengaman Biaya + Mount Drive + Kredensial Kaggle (Cache)

**Pengaman biaya** dipasang PALING AWAL: bila ada sel yang gagal saat Run all (termasuk mount Drive / kaggle.json), error ditampilkan lalu runtime dilepas otomatis (jeda 60 detik; tekan Stop untuk menahan).


In [ ]:
# ---- PENGAMAN BIAYA: bila ada sel GAGAL (exception) saat "Run all", tampilkan error lalu lepas runtime ----
# Tanpa ini, satu error sebelum/sesudah training membuat GPU menganggur (tetap ditagih) sampai Colab memutus sendiri.
# Menekan tombol Stop (KeyboardInterrupt) TIDAK memicu pengaman ini. Ada jeda 60 detik: tekan Stop di jeda itu bila
# ingin menahan runtime. Set False bila sedang men-debug interaktif.
AUTO_UNASSIGN_ON_ERROR = True

def _lepas_runtime_bila_error(shell, etype, evalue, tb, tb_offset=None):
    stb = shell.InteractiveTB.structured_traceback(etype, evalue, tb, tb_offset=tb_offset)
    shell._showtraceback(etype, evalue, stb)          # tampilkan error-nya dulu
    if shell.user_ns.get("AUTO_UNASSIGN_ON_ERROR", True):
        print("\n⛔ Sel GAGAL -- runtime dilepas otomatis dalam 60 detik agar kuota Colab tidak terbuang.\n"
              "   Checkpoint & results.csv yang sudah ada aman di Drive; perbaiki penyebabnya lalu jalankan ulang notebook.\n"
              "   (Tekan Stop sekarang bila ingin menahan runtime.)", flush=True)
        import time
        try:
            time.sleep(60)
        except KeyboardInterrupt:   # Stop ditekan di jeda -> runtime DITAHAN, pengaman tetap terpasang
            print("Dibatalkan (Stop): runtime TIDAK dilepas. Pengaman tetap aktif untuk sel berikutnya.", flush=True)
            return None
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as _e:
            print(f"Gagal melepas runtime otomatis ({_e}) -- matikan manual: Runtime > Disconnect and delete runtime")
    return None

get_ipython().set_custom_exc((Exception,), _lepas_runtime_bila_error)
print("Pengaman biaya aktif: runtime dilepas otomatis bila ada sel yang gagal.")

from google.colab import drive
drive.mount('/content/drive')

import os, stat, shutil

DRIVE_ROOT = "/content/drive/MyDrive/demamba-yolowd"
os.makedirs(DRIVE_ROOT, exist_ok=True)
KAGGLE_JSON_CACHE = f"{DRIVE_ROOT}/kaggle.json"

os.makedirs("/root/.kaggle", exist_ok=True)
assert os.path.exists(KAGGLE_JSON_CACHE), (
    f"kaggle.json tidak ada di {KAGGLE_JSON_CACHE} -- jalankan fase-setup-colab-mamba.ipynb dulu."
)
shutil.copy(KAGGLE_JSON_CACHE, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", stat.S_IRUSR | stat.S_IWUSR)
print("kaggle.json dimuat dari cache Drive.")

## 2. Tarik Dataset Steel-pipe + deform_ss2d.py dari Kaggle + Tripwire (sha256 & isi dataset) + Deteksi GPU


In [ ]:
import subprocess, sys, glob, torch

# TF32 dimatikan (konsisten dgn semua run proyek sejak B2 seed 1).
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print(f"TF32 dimatikan (matmul={torch.backends.cuda.matmul.allow_tf32}, cudnn={torch.backends.cudnn.allow_tf32})")

# ---------- Cek GPU PALING AWAL (sebelum unduhan) -- GPU salah = berhenti dalam hitungan detik ----------
_gpu = torch.cuda.get_device_name(0)
print(f"GPU: {_gpu}  (compute cap {torch.cuda.get_device_capability(0)})")
# Semua run grid-fix sebelumnya (P1 penuh steel-pipe, SWRD v1) memakai A100. GPU lain = numerik & waktu berbeda
# (T4 ±3x lebih lambat -> biaya naik). Ubah ke True HANYA bila sengaja, dan catat di tracker.
ALLOW_NON_A100 = False
assert "A100" in _gpu or ALLOW_NON_A100, (
    f"GPU = {_gpu}, BUKAN A100. Runtime > Change runtime type > A100, lalu Run all lagi "
    f"(atau set ALLOW_NON_A100 = True dengan sengaja).")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/deform_src", exist_ok=True)

STEEL_DIR = "/content/data/steelpipe"
if os.path.exists(STEEL_DIR):
    shutil.rmtree(STEEL_DIR)   # selalu unduh bersih
os.makedirs(STEEL_DIR)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/dsf-yolo-steel-tube-dataset",
                "-p", STEEL_DIR, "--unzip"], check=True)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/deform-ss2d-py",
                "-p", "/content/deform_src", "--unzip"], check=True)

DEFORM_SRC = glob.glob("/content/deform_src/**/deform_ss2d.py", recursive=True)[0]

# ---------- TRIPWIRE VERSI deform_ss2d.py (sha256) -- HARUS grid-fix ----------
#   cfed78de317a7781... = PRA-perbaikan (grid +0.5 px)   -> DITOLAK
#   675445871a2e4d55... = GRID-FIX                        -> DIPAKAI DI SINI (= P1 penuh steel-pipe & SWRD v1)
#   51520b33f9f77f28... = v2 jalur-terbatas (Fase 12)     -> DITOLAK
import hashlib
DEFORM_SHA256_EXPECTED = "675445871a2e4d55400175ef5f6fda83c90ece651bf818e3f54481bc14a6d9aa"
_raw = open(DEFORM_SRC, "rb").read()
DEFORM_SHA256 = hashlib.sha256(_raw).hexdigest()
_norm = hashlib.sha256(_raw.replace(b"\r\n", b"\n")).hexdigest()
_txt = _raw.decode("utf-8", "replace")
print(f"deform_ss2d.py sha256 = {DEFORM_SHA256}")
if "linspace(0.5, H - 0.5" in _txt or "linspace(0.5, N - 0.5" in _txt:
    raise RuntimeError("deform_ss2d.py = versi PRA-PERBAIKAN (grid +0.5 px). Unggah versi grid-fix ke Kaggle 'deform-ss2d-py'.")
if "bounded_path_positions" in _txt:
    raise RuntimeError("deform_ss2d.py = versi v2 jalur-terbatas (Fase 12). Bukan yang dipakai di sini.")
assert DEFORM_SHA256 == DEFORM_SHA256_EXPECTED or _norm == DEFORM_SHA256_EXPECTED, (
    f"deform_ss2d.py BUKAN versi grid-fix (sha256 {DEFORM_SHA256[:16]}..., harus 675445871a2e4d55...). STOP.")
DEFORM_SHA256 = DEFORM_SHA256_EXPECTED   # yang dicatat di CSV = sha berkas SUMBER (sebelum patch runtime NoProj)
print("Tripwire versi: deform_ss2d.py = GRID-FIX (675445871a2e4d55...) -- OK")

# ---------- Dataset steel-pipe: cari root YOLO + TRIPWIRE isi ----------
# Root YOLO = folder yang punya images/train2021 DAN labels/train2021 (= .../steel-tube-dataset-all/yolo, sama dgn
# fase-0-clean). Bila rilis memuat format lain dgn nama split sama, folder 'yolo' didahulukan.
_cands = sorted((p for p in glob.glob(f"{STEEL_DIR}/**/images/train2021", recursive=True)
                 if os.path.isdir(os.path.join(os.path.dirname(os.path.dirname(p)), "labels", "train2021"))),
                key=lambda p: ("/yolo/" not in p, p.count("/"), p))
assert _cands, f"Folder images/train2021 + labels/train2021 tidak ditemukan di {STEEL_DIR} -- cek hasil unduhan Kaggle"
if len(_cands) > 1:
    print("  (beberapa kandidat root dataset; dipilih yang pertama):", _cands)
STEEL_BASE = os.path.dirname(os.path.dirname(_cands[0]))
print(f"STEEL_BASE = {STEEL_BASE}")
STEEL_NAMES = ["air-hole", "bite-edge", "broken-arc", "crack", "dense-gas-hole", "non-fusion", "slag-inclusion", "unfused-hole"]
_IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
EXPECTED_STEEL = {"train2021": 2726, "val2021": 682}   # jumlah citra (rilis Yang et al. 2021, diverifikasi Sesi 3)
_tot_obj, _val_obj = 0, 0
for _sp, _n in EXPECTED_STEEL.items():
    _imgs = [p for p in glob.glob(f"{STEEL_BASE}/images/{_sp}/*") if os.path.splitext(p)[1].lower() in _IMG_EXT]
    _stems = {os.path.splitext(os.path.basename(p))[0] for p in _imgs}
    _txt_all = glob.glob(f"{STEEL_BASE}/labels/{_sp}/*.txt")
    _labs = [p for p in _txt_all if os.path.splitext(os.path.basename(p))[0] in _stems]   # abaikan classes.txt dsb.
    if len(_txt_all) != len(_labs):
        print(f"  ({_sp}: {len(_txt_all) - len(_labs)} berkas .txt tanpa citra pasangan diabaikan, mis. classes.txt)")
    assert len(_imgs) == _n, f"{_sp}: {len(_imgs)} citra, harus {_n}"
    _k = 0
    for _p in _labs:
        for _l in open(_p).read().splitlines():
            if _l.strip():
                _r = _l.split()
                assert len(_r) == 5 and 0 <= int(_r[0]) < 8, f"label rusak: {_p}"
                _k += 1
    _tot_obj += _k
    if _sp == "val2021":
        _val_obj = _k
    print(f"Tripwire {_sp}: {len(_imgs)} citra, {len(_labs)} berkas label, {_k} objek")
assert _tot_obj == 6783 and _val_obj == 1381, f"objek total {_tot_obj} (harus 6783), val {_val_obj} (harus 1381)"
for _c in glob.glob(f"{STEEL_BASE}/labels/*.cache"):
    os.remove(_c)   # cache label lama dari unduhan -- dibuat ulang oleh Ultralytics
print("Tripwire dataset steel-pipe: COCOK (2.726 + 682 citra, 6.783 objek, 1.381 di val)")


## 3. 🔧 KONFIGURASI — Edit MODEL / SEED / RUN_MODE Setiap Run


In [ ]:
# ============================================================
#  UBAH 3 BARIS INI SEBELUM "Run all"
# ============================================================
MODEL    = "P1-NoProj"   # "P1-NoProj" | "BackboneOnly" | "NeckOnly" | "BB-Late" | "BB-Early" | "BB-Late-Alt"
                         # "P1" HANYA untuk RUN_MODE probe (P1 penuh 5 seed grid-fix SUDAH ADA -- tidak diulang)
SEED     = 1             # 1-5 untuk run utama; probe HANYA seed 1
RUN_MODE = "utama"       # "utama" | "probe_adamw" | "probe_lr0.01"
# ============================================================

TARGET_EPOCHS = 150
USE_COS_LR = False
if RUN_MODE == "utama":
    OPTIMIZER, LR0 = "SGD", 0.002     # resep terkunci (§4.3)
elif RUN_MODE == "probe_adamw":
    OPTIMIZER, LR0 = "auto", 0.002    # default framework: optimizer=auto -> AdamW, LR otomatis (lr0 DIABAIKAN
                                      # framework). Satu-satunya setelan yang berubah = optimizer (spt run B2-auto §5.6).
elif RUN_MODE == "probe_lr0.01":
    OPTIMIZER, LR0 = "SGD", 0.01      # LR default SGD framework; selain itu identik resep studi.
else:
    raise ValueError(f"RUN_MODE tidak dikenal: {RUN_MODE}")

# Target params DIHITUNG PADA nc=8 (dataset steel-pipe) -- dicek PERSIS di pre-flight (Sel 5).
MODEL_CONFIG = {
    "P1":           {"yaml": "DeMamba-YOLO-T.yaml",               "params": 5_930_869, "ndef": 10},
    "P1-NoProj":    {"yaml": "DeMamba-YOLO-T-noproj.yaml",        "params": 6_329_230, "ndef": 10},
    "BackboneOnly": {"yaml": "DeMamba-YOLO-T-backbone-only.yaml", "params": 5_952_204, "ndef": 6},
    "NeckOnly":     {"yaml": "DeMamba-YOLO-T-neck-only.yaml",     "params": 5_965_577, "ndef": 4},
    "BB-Late":      {"yaml": "DeMamba-YOLO-T-bb-late.yaml",       "params": 5_964_057, "ndef": 4},
    "BB-Early":     {"yaml": "DeMamba-YOLO-T-bb-early.yaml",      "params": 5_975_059, "ndef": 2},
    "BB-Late-Alt":  {"yaml": "DeMamba-YOLO-T-bb-late-alt.yaml",   "params": 5_972_944, "ndef": 3},
}
assert MODEL in MODEL_CONFIG, f"MODEL tidak dikenal: {MODEL}"
if RUN_MODE == "utama":
    assert MODEL != "P1", "P1 penuh (grid-fix, 5 seed) SUDAH ADA -- tidak diulang. Pakai P1 hanya utk probe."
    assert SEED in (1, 2, 3, 4, 5), f"SEED harus 1-5, dapat: {SEED}"
else:
    assert MODEL == "P1" and SEED == 1, "Probe resep HANYA P1 seed 1"

DATASET = "steelpipe"
if RUN_MODE == "utama":
    RUN_NAME = f"gf_{MODEL.lower()}_seed{SEED}_150ep_lr{LR0}"
    CSV_TAG  = "steelpipe_gf"
    OUT_CSV_NAME = "per_class_results_gridfix.csv"
elif RUN_MODE == "probe_adamw":
    RUN_NAME = f"gf_probe_adamw_p1_seed{SEED}_150ep"
    CSV_TAG  = "steelpipe_gf_probe_adamw"
    OUT_CSV_NAME = "per_class_results_gridfix_probe.csv"
else:
    RUN_NAME = f"gf_probe_p1_seed{SEED}_150ep_lr{LR0}"
    CSV_TAG  = "steelpipe_gf_probe_lr0.01"
    OUT_CSV_NAME = "per_class_results_gridfix_probe.csv"
assert RUN_NAME.startswith("gf_"), "semua run fase ini WAJIB berawalan gf_ (cegah tabrakan dgn folder lama)"

CONFIG_YAML     = MODEL_CONFIG[MODEL]["yaml"]
EXPECTED_PARAMS = MODEL_CONFIG[MODEL]["params"]
EXPECTED_NDEF   = MODEL_CONFIG[MODEL]["ndef"]
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/analysis", exist_ok=True)
OUT_CSV    = f"{DRIVE_ROOT}/analysis/{OUT_CSV_NAME}"
STATUS_CSV = f"{DRIVE_ROOT}/analysis/status_latih_gridfix.csv"

print(f"MODEL={MODEL}  SEED={SEED}  RUN_MODE={RUN_MODE}  RUN_NAME={RUN_NAME}")
print(f"optimizer={OPTIMIZER}  lr0={LR0}  cos_lr={USE_COS_LR}  epochs={TARGET_EPOCHS}")
print(f"CSV: {OUT_CSV}  (tag {CSV_TAG})")
if RUN_MODE != "utama":
    print(">>> MODE PROBE RESEP -- BUKAN run perbandingan. Dibaca dgn aturan no. 4 di Sel 0. <<<")


## 4a. Clone Mamba-YOLO + Catat Commit


In [ ]:
import re, py_compile

REPO = "/content/Mamba-YOLO"
# WAJIB sebelum rmtree: pastikan cwd proses TIDAK sedang di dalam REPO (bisa terjadi kalau
# notebook di-run ulang di kernel yang sama setelah sel pre-flight sempat os.chdir(REPO) --
# menghapus folder yang sedang "ditempati" proses bikin SEMUA subprocess berikutnya gagal
# dgn "fatal: Unable to read current working directory").
os.chdir("/content")
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/HZAI-ZJNU/Mamba-YOLO.git", REPO], check=True)

DEFORM = f"{REPO}/ultralytics/nn/modules/deform_ss2d.py"
shutil.copy(DEFORM_SRC, DEFORM)

src = open(DEFORM).read()
checks = {
    "STE (gradien Delta-p)": "x_soft" in src,
    "ssm_ratio=1.4 (capacity match P1)": "ssm_ratio=1.4" in src,
    "subclass block terdaftar": "class DeformVSSBlock" in src and "class DeformXSSBlock" in src,
}
if not all(checks.values()):
    raise RuntimeError("deform_ss2d.py BUKAN versi final -- cek dataset 'deform-ss2d-py' di Kaggle.")
print("Tripwire deform_ss2d.py OK")

REPO_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
REPO_COMMIT_KNOWN = "51f7be9bd5ab907b6f7d2c91bb71ca5a4be063aa"   # commit HZAI-ZJNU/Mamba-YOLO yang tercatat (21 Okt 2026)
print(f"Mamba-YOLO commit: {REPO_COMMIT}")
if REPO_COMMIT != REPO_COMMIT_KNOWN:
    print("⚠️  Commit upstream BERBEDA dari yang tercatat -- patch di Sel 4b tetap dicek ketat (assert 1x); "
          "catat commit ini di tracker sebelum melanjutkan.")


## 4b. Patch Registrasi + Patch P1-NoProj + YAML Semua Varian + YAML Dataset

Patch registrasi identik notebook SWRD v1/Fase 2, ditambah NP-1…NP-4 (`PATCH-p1-noproj.md`) yang diterapkan ke salinan grid-fix, dan patch 6.1 (`np.trapz` → `np.trapezoid`, numpy ≥ 2.4). Sel ini tidak memakai GPU/jaringan (diuji offline sebelum dikirim).


In [ ]:
import re, py_compile
def patch(path, subs):
    p = f"{REPO}/{path}"
    src = open(p).read()
    for old, new, tag in subs:
        if new in src:
            print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new)
        print(f"  OK {tag}")
    open(p, "w").write(src)

print("\n### Patch")
patch("ultralytics/nn/modules/mamba_yolo.py", [
    ("class XSSBlock(nn.Module):\n    def __init__(",
     "class XSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.1"),
    ("nn.Sequential(*(SS2D(d_model=self.hidden_dim,",
     "nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,", "1.2"),
    ("class VSSBlock(nn.Module):\n    def __init__(",
     "class VSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.3"),
    ("            self.op = SS2D(",
     "            self.op = self.SSM_CLASS(", "1.4"),
])
patch("ultralytics/nn/modules/deform_ss2d.py", [
    ("from .mamba_yolo import SS2D",
     "from .mamba_yolo import SS2D, VSSBlock, XSSBlock", "2.1"),
])

# ---- Patch NP-1..NP-5: varian P1-NoProj (PATCH-p1-noproj.md), diterapkan RUNTIME ke salinan grid-fix ----
# Semua argumen baru default ke perilaku lama -> P1 & 5 varian parsial TIDAK berubah (dicek di pre-flight Sel 5:
# P1 tetap 5.930.869 params). sha256 yang dicatat di CSV = berkas sumber grid-fix SEBELUM patch ini.
patch("ultralytics/nn/modules/deform_ss2d.py", [
    ("    def __init__(self, in_ch: int, kernel_size: int):\n"
     "        super().__init__()\n"
     "        pad = kernel_size // 2 if kernel_size > 1 else 0\n"
     "        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1)\n",
     "    def __init__(self, in_ch: int, kernel_size: int, use_proj: bool = True):\n"
     "        super().__init__()\n"
     "        pad = kernel_size // 2 if kernel_size > 1 else 0\n"
     "        self.use_proj = use_proj\n"
     "        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1) if use_proj else nn.Identity()\n", "NP-1"),
    ("    def __init__(self, embed_dim: int, kernel_size: int = 5):\n"
     "        super().__init__()\n"
     "        self.conv_offset = ConvOffset(embed_dim, kernel_size)\n",
     "    def __init__(self, embed_dim: int, kernel_size: int = 5, use_proj: bool = True):\n"
     "        super().__init__()\n"
     "        self.conv_offset = ConvOffset(embed_dim, kernel_size, use_proj=use_proj)\n", "NP-2"),
    ("                 deform_kernel_size: int = None,\n"
     "                 **kwargs):\n",
     "                 deform_kernel_size: int = None,\n"
     "                 deform_use_proj: bool = True,\n"
     "                 **kwargs):\n", "NP-3a"),
    ("        self.deform_scans = nn.ModuleList([\n"
     "            DeformableScan(d_inner, kernel_size=ks) for _ in range(n_deform)\n"
     "        ])\n",
     "        self.deform_use_proj = deform_use_proj\n"
     "        self.deform_scans = nn.ModuleList([\n"
     "            DeformableScan(d_inner, kernel_size=ks, use_proj=deform_use_proj) for _ in range(n_deform)\n"
     "        ])\n", "NP-3b"),
    ('\nif __name__ == "__main__":',
     '\n\nclass DeformVSSBlockNP(VSSBlock):\n'
     '    """P1-NoProj (backbone): ssm_ratio TETAP dari YAML/VSSBlock (2.0) + ConvOffset tanpa proj.\n'
     '    Total model 6.329.230 params (+5,72% vs B2) -- sengaja di atas toleransi +/-5% (stress test)."""\n'
     '    SSM_CLASS = partial(DeformSS2D, deform_mode="k3_replace", deform_kernel_size=5, deform_use_proj=False)\n'
     '    # TIDAK meng-override __init__ (beda dgn DeformVSSBlock yg memaksa ssm_ratio=1.4)\n'
     '\n\nclass DeformXSSBlockNP(XSSBlock):\n'
     '    """P1-NoProj (neck). Lihat DeformVSSBlockNP."""\n'
     '    SSM_CLASS = partial(DeformSS2D, deform_mode="k3_replace", deform_kernel_size=5, deform_use_proj=False)\n'
     '\n\nif __name__ == "__main__":', "NP-4"),
])
patch("ultralytics/nn/tasks.py", [
    ("    XSSBlock\n)\n",
     "    XSSBlock\n)\nfrom ultralytics.nn.modules.deform_ss2d import DeformVSSBlock, DeformXSSBlock, DeformVSSBlockNP, DeformXSSBlockNP\n", "3.1"),
    ("SimpleStem, VisionClueMerge, VSSBlock, XSSBlock\n        }:",
     "SimpleStem, VisionClueMerge, VSSBlock, XSSBlock, DeformVSSBlock, DeformXSSBlock, DeformVSSBlockNP, DeformXSSBlockNP\n        }:", "3.2"),
    ("RepC3, XSSBlock}:",
     "RepC3, XSSBlock, DeformXSSBlock, DeformXSSBlockNP}:", "3.3"),
    ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
     '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:', "4.1"),
    ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
     '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)', "4.2"),
])
patch("ultralytics/utils/torch_utils.py", [
    ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
     '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:', "5.1"),
])

y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
y = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", y)
y = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", y)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml", "w").write(y)

# Ablasi B: 2 varian campuran -- backbone-saja-deformable, neck-saja-deformable.
# Split di penanda "# Mamba-YOLO PAFPN" (awal section head/neck), regex-swap CUMA
# di satu bagian. Params sudah diverifikasi offline: BackboneOnly=5.952M(-0.58%),
# NeckOnly=5.966M(-0.36%) vs B2 -- keduanya dlm toleransi +-5% TANPA ssm_ratio baru.
base_yaml = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
split_marker = "# Mamba-YOLO PAFPN"
idx = base_yaml.find(split_marker)
assert idx != -1, "penanda split YAML tidak ketemu -- struktur Mamba-YOLO-T.yaml berubah?"
backbone_part, head_part = base_yaml[:idx], base_yaml[idx:]
assert "VSSBlock" in backbone_part and "XSSBlock" not in backbone_part
assert "XSSBlock" in head_part and "VSSBlock" not in head_part

backbone_deform = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", backbone_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-backbone-only.yaml", "w").write(
    backbone_deform + head_part)

head_deform = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", head_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-neck-only.yaml", "w").write(
    backbone_part + head_deform)
print("  OK 2 YAML Ablasi B (backbone-only, neck-only) berhasil digenerate")

# Ablasi B-lanjut: 3 varian stage-wise di backbone (literatur-deformable-lintas-paradigma.md
# SS6.3). Neck TIDAK disentuh (tetap XSSBlock biasa di semua 3 varian ini) -- isolasi bersih
# dari pertanyaan Ablasi B sebelumnya. Tiap baris backbone punya komentar posisi unik
# ("# 1","# 3","# 5","# 7") dipakai utk menarget baris tepat, assert-guarded (fail loud
# kalau match != 1x, mis. krn upstream ubah format YAML).
def _swap_positions(text, positions):
    lines = text.split("\n")
    hit = {p: 0 for p in positions}
    for i, line in enumerate(lines):
        stripped = line.rstrip()
        for p in positions:
            if stripped.endswith(f"# {p}") and "VSSBlock" in line and "Deform" not in line:
                lines[i] = line.replace("VSSBlock", "DeformVSSBlock")
                hit[p] += 1
    for p in positions:
        assert hit[p] == 1, f"posisi {p}: ditemukan {hit[p]}x (harus tepat 1x) -- format YAML berubah?"
    return "\n".join(lines)

stagewise_variants = {"bb-deep": [7], "bb-late": [5, 7], "bb-early": [1, 3]}
for vname, positions in stagewise_variants.items():
    result = _swap_positions(base_yaml, positions)
    open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-{vname}.yaml", "w").write(result)
print(f"  OK {len(stagewise_variants)} YAML Ablasi B-lanjut (bb-deep, bb-late, bb-early) berhasil digenerate")

# BB-Late-Alt (BERSYARAT, backlog Section 2d) -- sumbu kerapatan intra-stage. Posisi 5
# dipecah jadi 3 baris terpisah (D-S-D) krn parse_model bikin n INSTANCE TERPISAH dari
# kelas yg sama per baris (bukan loop internal) -- lihat tasks.py baris ~946. Memecah 1
# baris jadi 3 menggeser SEMUA indeks "from" eksplisit di bawahnya +2 -- WAJIB diperbaiki,
# assert-guarded (kalau salah, parse_model gagal keras karena channel Concat tak cocok,
# bukan diam-diam salah). Sudah diverifikasi offline: save-layers hasil parse_model =
# [3,7,10,13,16,19,22], cocok persis perhitungan manual.
_lines = base_yaml.split("\n")
_t5 = None
for _i, _l in enumerate(_lines):
    if _l.rstrip().endswith("# 5") and "VSSBlock" in _l and "Deform" not in _l:
        assert _t5 is None, "posisi 5: >1 baris cocok"
        _t5 = _i
assert _t5 is not None, "posisi 5 tidak ditemukan"
_dsd = [
    "  - [-1, 1, DeformVSSBlock, [512]]      # 5a (D)",
    "  - [-1, 1, VSSBlock, [512]]             # 5b (S)",
    "  - [-1, 1, DeformVSSBlock, [512]]      # 5c (D)",
]
_lines = _lines[:_t5] + _dsd + _lines[_t5+1:]
_hit7 = 0
for _i, _l in enumerate(_lines):
    if _l.rstrip().endswith("# 7") and "VSSBlock" in _l and "Deform" not in _l:
        _lines[_i] = _l.replace("VSSBlock", "DeformVSSBlock")
        _hit7 += 1
assert _hit7 == 1, f"posisi 7: {_hit7}x"
_text = "\n".join(_lines)
_reindex = [
    ("  - [[-1, 5], 1, Concat, [1]]  # cat backbone P4",
     "  - [[-1, 7], 1, Concat, [1]]  # cat backbone P4 (geser +2 krn split posisi 5)"),
    ("  - [[-1, 11], 1, Concat, [1]]  # cat head P4",
     "  - [[-1, 13], 1, Concat, [1]]  # cat head P4 (geser +2 krn split posisi 5)"),
    ("  - [[-1, 8], 1, Concat, [1]]  # cat head P5",
     "  - [[-1, 10], 1, Concat, [1]]  # cat head P5 (geser +2 krn split posisi 5)"),
    ("  - [[14, 17, 20], 1, Detect, [nc]]  # Detect(P3, P4, P5)",
     "  - [[16, 19, 22], 1, Detect, [nc]]  # Detect(P3, P4, P5) (geser +2 krn split posisi 5)"),
]
for _old, _new in _reindex:
    assert _text.count(_old) == 1, f"GAGAL cari saat reindex: {_old!r}"
    _text = _text.replace(_old, _new)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-bb-late-alt.yaml", "w").write(_text)
print("  OK 1 YAML BB-Late-Alt (D-S-D, bersyarat) berhasil digenerate, indeks head diperbaiki")

# ---- Patch 6.1: np.trapz DIHAPUS mulai numpy 2.4.0; Ultralytics 8.2.29 memakainya utk menghitung AP ----
# Tanpa patch ini, bila Colab sudah memakai numpy >= 2.4, training CRASH di validasi epoch pertama.
# np.trapezoid = fungsi yang SAMA (nama baru sejak numpy 2.0; np.trapz di 2.0-2.3 hanya memanggil np.trapezoid)
# -> angka mAP IDENTIK dgn run sebelumnya. Fallback np.trapz hanya utk numpy 1.x.
patch("ultralytics/utils/metrics.py", [
    ("ap = np.trapz(np.interp(x, mrec, mpre), x)",
     "ap = (np.trapezoid if hasattr(np, 'trapezoid') else np.trapz)(np.interp(x, mrec, mpre), x)", "6.1"),
])
import numpy as _np
print(f"  numpy {_np.__version__}: np.trapz {'ada' if hasattr(_np, 'trapz') else 'TIDAK ADA (patch 6.1 mencegah crash)'}")

# P1-NoProj: posisi deformable SAMA dgn P1 (1,3,5,7,11,14,17,20), kelas *NP (ssm_ratio 2.0, tanpa proj)
_y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml").read()
assert _y.count("DeformVSSBlock") == 4 and _y.count("DeformXSSBlock") == 4, "YAML P1 tak terduga"
_y = re.sub(r"\bDeformVSSBlock\b", "DeformVSSBlockNP", _y)
_y = re.sub(r"\bDeformXSSBlock\b", "DeformXSSBlockNP", _y)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-noproj.yaml", "w").write(_y)
print("  OK YAML P1-NoProj (DeMamba-YOLO-T-noproj.yaml) digenerate")

# data YAML steel-pipe dgn path ABSOLUT (nama kelas = nama lama; pemetaan naskah ada di Sel 0)
import yaml as _yaml
STEEL_YAML = f"{REPO}/ultralytics/cfg/datasets/steelpipe.yaml"
with open(STEEL_YAML, "w") as _f:
    _yaml.safe_dump({"path": STEEL_BASE, "train": "images/train2021", "val": "images/val2021",
                     "nc": 8, "names": STEEL_NAMES}, _f, sort_keys=False, allow_unicode=True)
print(f"  OK steelpipe.yaml ditulis: {STEEL_YAML}")
print(open(STEEL_YAML).read())

for f in ["ultralytics/nn/modules/mamba_yolo.py", "ultralytics/nn/modules/deform_ss2d.py",
          "ultralytics/nn/tasks.py", "ultralytics/utils/torch_utils.py", "ultralytics/utils/metrics.py"]:
    py_compile.compile(f"{REPO}/{f}", doraise=True)
print("  OK sintaks semua file valid -- SEMUA PATCH + YAML OK")


## 4c. Build Kernel `selective_scan` + Install


In [ ]:
import re
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja"], check=True)
nproc = int(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip() or "2")
os.environ["MAX_JOBS"] = str(max(1, nproc))

# Cache wheel selective_scan ke Drive. KUNCI CACHE WAJIB memuat versi CUDA torch:
# torch "2.11.0" bisa dibangun utk CUDA 12 (cu128) maupun CUDA 13 (cu130) dgn nomor versi SAMA.
# Kunci lama hanya "torch2.11.0" -> wheel hasil compile CUDA 12 (sesi lalu) dipasang ke torch CUDA 13
# -> ImportError libcudart.so.12. Kunci baru memisahkan keduanya; wheel lama tetap di Drive (tak dipakai).
import shutil as _sh
cap = torch.cuda.get_device_capability(0)
_tv = torch.__version__.replace("+", "-")
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{_tv}_cuda{torch.version.cuda}_py{sys.version_info.major}{sys.version_info.minor}"
SS_WHEEL_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"
os.makedirs(SS_WHEEL_DIR, exist_ok=True)

def _nvcc_version():
    nvcc = os.path.join(os.environ.get("CUDA_HOME", "/usr/local/cuda"), "bin", "nvcc")
    if not os.path.exists(nvcc):
        nvcc = _sh.which("nvcc") or nvcc
    try:
        out = subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout
        return re.search(r"release (\d+)\.(\d+)", out).group(0).split()[-1]
    except Exception:
        return None

def _ss_import_ok():
    # dicek di proses TERPISAH: import yang gagal tidak meninggalkan modul rusak di kernel ini
    r = subprocess.run([sys.executable, "-c", "import torch, selective_scan_cuda_core"], capture_output=True, text=True)
    return r.returncode == 0, (r.stderr.strip().splitlines() or [""])[-1]

def _ss_install(wheels):
    # --force-reinstall --no-deps: nama & versi wheel sama dgn yg (mungkin) sudah terpasang -> tanpa ini pip melewatinya
    subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps"] + wheels, check=True)

print(f"torch {torch.__version__} | CUDA torch {torch.version.cuda} | nvcc {_nvcc_version()} | {SS_TAG}")
cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
# Wheel LAMA (kunci tanpa versi CUDA, mis. 'sm80_torch2.11.0_py313') dibuat di sesi torch 2.11.0+cu128.
# Hanya dipakai bila torch sekarang juga build cu128, dan tetap WAJIB lolos uji import di bawah.
_legacy = f"{DRIVE_ROOT}/wheels/selective_scan_sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
if not cached_ss and torch.__version__.endswith("+cu128") and glob.glob(f"{_legacy}/*.whl"):
    for _w in glob.glob(f"{_legacy}/*.whl"):
        _sh.copy(_w, SS_WHEEL_DIR)
    cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
    print(f"Wheel lama ditemukan di {os.path.basename(_legacy)} (cu128) -- disalin ke kunci baru, akan diuji import")
if cached_ss:
    print(f"Wheel selective_scan cache ditemukan ({SS_TAG}) -- install langsung, skip compile")
    _ss_install(cached_ss)
    ok, err = _ss_import_ok()
    if not ok:
        print(f"  Wheel cache GAGAL di-import ({err}) -- dihapus, compile ulang")
        for w in cached_ss: os.remove(w)
        cached_ss = []
if not cached_ss:
    _nv = _nvcc_version()
    assert _nv is not None, "nvcc tidak ditemukan -- tidak bisa compile selective_scan."
    assert _nv.split(".")[0] == str(torch.version.cuda).split(".")[0], (
        f"CUDA MAJOR berbeda: torch dibangun utk CUDA {torch.version.cuda}, nvcc sistem {_nv}. "
        f"Tidak ada wheel cache utk GPU ini ({SS_TAG}). Pakai GPU A100 (sama dgn run sebelumnya) atau kirim pesan ini ke Claude.")
    # CUDA 13 menghapus dukungan arsitektur < sm_75 (Maxwell/Pascal/Volta). setup.py selective_scan
    # (warisan VMamba) meng-compile sm_70 -> nvcc 13 menolak 'compute_70'. Ganti ke sm_75 HANYA bila CUDA>=13.
    # Tidak mengubah numerik di A100: kernel yg dipakai GPU sm_80 tetap cubin sm_80. (sm_75 = T4.)
    _setup = f"{REPO}/selective_scan/setup.py"
    if int(str(torch.version.cuda).split(".")[0]) >= 13:
        _ss = open(_setup).read()
        _n70 = len(re.findall(r"compute_70\s*,\s*code\s*=\s*sm_70", _ss))
        if _n70:
            _ss = re.sub(r"compute_70\s*,\s*code\s*=\s*sm_70", "compute_75,code=sm_75", _ss)
            open(_setup, "w").write(_ss)
        print(f"  setup.py selective_scan: {_n70} target sm_70 diganti sm_75 (CUDA {torch.version.cuda}); "
              f"arch tersisa: {sorted(set(re.findall(r'sm_(\d+)', _ss)))}")
    print(f"Belum ada cache utk {SS_TAG} -- compile sekali ini (~beberapa menit), lalu disimpan ke Drive")
    subprocess.run([sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-v", "-w", SS_WHEEL_DIR],
                   cwd=f"{REPO}/selective_scan", check=True)
    new_ss_wheels = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
    _ss_install(new_ss_wheels)
    ok, err = _ss_import_ok()
    assert ok, f"selective_scan hasil compile tetap gagal di-import: {err}"
    print(f"Tersimpan ke {SS_WHEEL_DIR} -- sesi berikutnya dgn kombinasi GPU/torch/CUDA yg sama langsung cepat")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)

for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]:
    del sys.modules[k]
import selective_scan_cuda_core
print("selective_scan_cuda_core berhasil di-import.")
ENV_INFO = dict(repo_commit=REPO_COMMIT, torch=torch.__version__, cuda=torch.version.cuda, nvcc=_nvcc_version(), python=sys.version.split()[0],
                gpu=torch.cuda.get_device_name(0), ss_tag=SS_TAG)
print("Lingkungan run (catat di tracker):", ENV_INFO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn", "thop", "timm", "einops"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                "wandb", "ray", "tensorboard", "tensorboardX", "tensorboard-data-server",
                "clearml", "comet_ml", "dvclive", "mlflow", "neptune"])

print("\nSEMUA PATCH + BUILD OK.")


## 5. Pre-flight — Jumlah Parameter PERSIS + Regresi P1


In [ ]:
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from ultralytics import settings
settings.update({"tensorboard": False})

from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules.deform_ss2d import DeformSS2D, ConvOffset
from ultralytics.nn.modules.mamba_yolo import SS2D
import torch.nn as nn

def _count(yaml_name):
    m = DetectionModel(f"ultralytics/cfg/models/mamba-yolo/{yaml_name}", ch=3, nc=8, verbose=False)
    n_par = sum(x.numel() for x in m.parameters())
    n_def = sum(1 for x in m.modules() if isinstance(x, DeformSS2D))
    n_noproj = sum(1 for x in m.modules() if isinstance(x, ConvOffset) and isinstance(x.proj, nn.Identity))
    n_co = sum(1 for x in m.modules() if isinstance(x, ConvOffset))
    del m
    return n_par, n_def, n_noproj, n_co

cfg_path = f"ultralytics/cfg/models/mamba-yolo/{CONFIG_YAML}"
n_par, n_def, n_np, n_co = _count(CONFIG_YAML)
print(f"MODEL={MODEL}  SEED={SEED}  RUN_MODE={RUN_MODE}  config={CONFIG_YAML}")
print(f"Params: {n_par:,} (target {EXPECTED_PARAMS:,})  DeformSS2D: {n_def} (target {EXPECTED_NDEF})  "
      f"ConvOffset tanpa proj: {n_np}/{n_co}")
assert n_par == EXPECTED_PARAMS, f"Params {n_par:,} != {EXPECTED_PARAMS:,}. STOP."
assert n_def == EXPECTED_NDEF, f"Jumlah DeformSS2D {n_def} != {EXPECTED_NDEF}. STOP."
if MODEL == "P1-NoProj":
    assert n_np == n_co == 10, f"P1-NoProj harus 10/10 ConvOffset tanpa proj, dapat {n_np}/{n_co}"
else:
    assert n_np == 0, f"{MODEL} tidak boleh punya ConvOffset tanpa proj ({n_np})"

# Regresi: patch NoProj tidak boleh mengubah P1 (5.930.869 params, semua proj ada)
_p1 = _count("DeMamba-YOLO-T.yaml")
assert _p1 == (5_930_869, 10, 0, 10), f"REGRESI P1 GAGAL: {_p1}"
print("Regresi P1 (5.930.869, proj utuh) OK")

torch.cuda.empty_cache()
print("Pre-flight LOLOS.")


## 6. Cek Status Checkpoint — Baru / Belum Selesai / Sudah Selesai

Sel ini yang membuat notebook aman dijalankan ulang. Tidak ada training di sini, cuma
pengecekan status.

In [ ]:
from pathlib import Path

last_ckpt = Path(DRIVE_OUTPUT) / RUN_NAME / "weights" / "last.pt"

if not last_ckpt.exists():
    STATUS = "BARU"
    print(f"STATUS: {STATUS} -- belum ada checkpoint, training akan mulai dari nol.")
else:
    raw = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    ep = raw.get("epoch", -1)
    has_opt = raw.get("optimizer") is not None
    if has_opt and ep >= 0 and ep < TARGET_EPOCHS - 1:
        STATUS = "BELUM_SELESAI"
        print(f"STATUS: {STATUS} -- checkpoint ada, epoch tersimpan={ep}, optimizer ada.")
        print(f"  -> Akan RESUME (genuine, LR schedule kontinu) dari epoch {ep+1}.")
    else:
        STATUS = "SELESAI"
        print(f"STATUS: {STATUS} -- checkpoint sudah final_eval (optimizer={has_opt}, epoch={ep}).")
        print("  -> Training TIDAK diulang. Lanjut ke Sel 8 untuk baca hasil.")

print(f"\nlast.pt: {last_ckpt}  (ada: {last_ckpt.exists()})")

## 7. Training — Mulai Baru atau Resume, Sesuai Status Sel 6

**`patience=0` WAJIB** — mematikan EarlyStopping. Proyek ini butuh jumlah epoch yang
**terkontrol ketat** (150, sama persis tiap run) untuk perbandingan B2 vs P1 yang adil.
Default Ultralytics `patience=100` membuat run berhenti otomatis begitu 100 epoch berlalu
tanpa perbaikan `fitness` — kalau dibiarkan, run yang berbeda bisa berhenti di jumlah epoch
berbeda-beda (satu model plateau lebih cepat dari yang lain), merusak prinsip
compute-matched/epoch-matched yang jadi dasar seluruh perbandingan proyek ini.

In [ ]:
if STATUS == "BARU":
    model = YOLO(cfg_path)
    model.train(
        data=STEEL_YAML,  # <- dataset steel-pipe (steelpipe.yaml, path absolut, Sel 4b)
        epochs=TARGET_EPOCHS,
        batch=8, imgsz=640, device=0, amp=True, seed=SEED,
        patience=0,   # WAJIB: matikan EarlyStopping, perbandingan butuh epoch terkontrol ketat
        cache=True,   # percepat: dataset muat di RAM. TIDAK mengubah data/hasil.
        optimizer=OPTIMIZER,  # "SGD" (run utama & probe_lr0.01) | "auto" (HANYA probe_adamw) -- lihat Sel 3
        cos_lr=USE_COS_LR,  # False (default eksperimen ini) -- kosinus TERBUKTI LEBIH BURUK,
                            # lihat catatan lengkap di Sel 3.
        lr0=LR0,  # 0.002 (resep terkunci); 0.01 HANYA probe_lr0.01
        # JANGAN PERNAH set time= -- itu override epochs= secara diam-diam (sama bahayanya dgn patience)
        project=DRIVE_OUTPUT, name=RUN_NAME,
        exist_ok=True,  # pakai folder RUN_NAME apa adanya -- JANGAN biarkan Ultralytics membuat "…2"
    )
elif STATUS == "BELUM_SELESAI":
    model = YOLO(str(last_ckpt))
    # TANPA epochs=/patience=/optimizer= -- resume mengambil train_args tersimpan di checkpoint
    # (150, patience=0, optimizer=SGD -- asalkan run aslinya sudah pakai settingan ini di atas).
    model.train(resume=True)
else:
    model = YOLO(str(last_ckpt))
    print("Training sudah selesai sebelumnya -- sel ini di-skip, tidak training ulang.")

SAVE_DIR = model.trainer.save_dir if hasattr(model, "trainer") and model.trainer else Path(DRIVE_OUTPUT) / RUN_NAME
print(f"\nSAVE_DIR: {SAVE_DIR}")

# Verifikasi status akhir -- kalau sesi ini SENDIRI terputus, baris ini tidak akan tercetak,
# dan menjalankan ulang notebook (dari Sel 1) akan mendeteksi STATUS=BELUM_SELESAI otomatis.
final_check = torch.load(str(Path(SAVE_DIR) / "weights" / "last.pt"), map_location="cpu", weights_only=False)
if final_check.get("optimizer") is None:
    print(f"Training {RUN_NAME} SELESAI PENUH ({TARGET_EPOCHS} epoch).")
else:
    print(f"Training belum selesai (epoch {final_check.get('epoch')}) -- jalankan ulang notebook ini untuk lanjut.")

## 8. Baca Hasil (Hanya Kalau Sudah Selesai Penuh)

In [ ]:
import pandas as pd

df = pd.read_csv(Path(SAVE_DIR) / "results.csv")
df.columns = df.columns.str.strip()
last = df.iloc[-1]

print(f"=== {RUN_NAME} — {len(df)} baris tercatat (epoch terakhir: {int(df['epoch'].max())}) ===")
print(f"  mAP50    = {last['metrics/mAP50(B)']:.4f}")
print(f"  mAP50:95 = {last['metrics/mAP50-95(B)']:.4f}")
print(f"  P={last['metrics/precision(B)']:.4f}  R={last['metrics/recall(B)']:.4f}")

# Cek epoch TERAKHIR yang tercapai, BUKAN jumlah baris -- results.csv kadang bolong di
# tengah (mis. Drive I/O sempat gagal nulis 1-2 baris) walau training-nya sendiri utuh
# tanpa gangguan. len(df) < TARGET_EPOCHS bisa salah nunjuk "belum selesai" padahal
# training sudah genap sampai epoch target -- ditemukan dari kasus B2 seed4 (146 baris
# tapi epoch max=150, gap di epoch 26-27 & 87-88, loss mulus tanpa lompatan di sekitarnya).
epoch_tercapai = int(df['epoch'].max())
n_gap = TARGET_EPOCHS - len(df)
if epoch_tercapai < TARGET_EPOCHS:
    print(f"\n⚠️  Epoch terakhir cuma {epoch_tercapai}/{TARGET_EPOCHS} -- training BELUM selesai.")
    print("   Jalankan ulang notebook ini (Runtime > Run all) untuk lanjut.")
else:
    print(f"\nSELESAI PENUH -- epoch terakhir tercapai {epoch_tercapai}/{TARGET_EPOCHS}.")
    if n_gap > 0:
        missing = sorted(set(range(1, TARGET_EPOCHS+1)) - set(df['epoch'].astype(int)))
        print(f"  Catatan: {n_gap} baris log bolong di tengah (epoch {missing}) -- kemungkinan")
        print(f"  Drive I/O sempat gagal nulis, BUKAN training terputus. Cek loss di sekitar")
        print(f"  epoch itu kalau ragu (harus mulus, tanpa lompatan/reset).")
    print(f"Checkpoint: {Path(SAVE_DIR) / 'weights' / 'best.pt'}")
    print("Catat path ini untuk arsip & analisis agregat multi-seed nanti.")


## 8b. Status Latih — Kriteria Kolaps Terkunci (§4.4)

Otomatis; tidak ada keputusan manual. Run kolaps **tidak diulang**.


In [ ]:
# ---- Sel 8b: STATUS LATIH menurut kriteria kolaps yang DIKUNCI (§4.4 naskah) ----
import numpy as np, pandas as pd
_r = pd.read_csv(Path(SAVE_DIR) / "results.csv", skipinitialspace=True)
_r.columns = [c.strip() for c in _r.columns]
for _c in _r.columns:
    _r[_c] = pd.to_numeric(_r[_c], errors="coerce")
assert int(_r["epoch"].max()) == TARGET_EPOCHS, "Run belum 150 epoch -- jalankan ulang notebook (resume) dulu."
_ep, _cls = _r["epoch"], _r["train/cls_loss"]
_tr = _r[["train/box_loss", "train/cls_loss", "train/dfl_loss"]].to_numpy()
_bad = ~np.isfinite(_tr).all(axis=1)
_e140 = int(_ep[_ep <= 140].max())   # 140, atau epoch terakhir <=140 bila baris log 140 bolong (Drive I/O)
_rasio = float(_cls[_ep == _e140].iloc[0] / _cls[_ep <= 140].min()) if not _bad.any() else float("nan")
if _bad.any():
    STATUS_LATIH, _alasan = "KOLAPS", f"loss latih non-finite sejak epoch {int(_ep[_bad].min())}"
elif _rasio >= 1.5:
    STATUS_LATIH, _alasan = "KOLAPS", f"cls_loss epoch {_e140} = {_rasio:.2f} x minimum epoch 1-140"
else:
    STATUS_LATIH, _alasan = "sehat", f"cls_loss epoch {_e140} = {_rasio:.2f} x minimum epoch 1-140"
_fit = 0.1 * _r["metrics/mAP50(B)"] + 0.9 * _r["metrics/mAP50-95(B)"]
_best_ep = int(_ep[_fit.idxmax()])
print(f"=== {RUN_NAME}: {STATUS_LATIH} -- {_alasan}; best.pt = epoch {_best_ep}" + (f" (epoch 140 bolong, dipakai {_e140})" if _e140 != 140 else ""))
if STATUS_LATIH == "KOLAPS":
    print("⚠️  Run KOLAPS: JANGAN diulang (aturan no. 2). Tetap jalankan Sel 9 -- angka checkpoint pra-kolaps dicatat apa adanya.")

_row = pd.DataFrame([{"dataset": CSV_TAG, "model": MODEL, "seed": SEED, "run_name": RUN_NAME,
                      "status": STATUS_LATIH, "rasio_cls_e140_min": _rasio, "epoch_rasio": _e140,
                      "nonfinite_epochs": int(_bad.sum()), "best_epoch": _best_ep,
                      "deform_sha16": DEFORM_SHA256[:16]}])
if Path(STATUS_CSV).exists():
    _old = pd.read_csv(STATUS_CSV)
    _row = pd.concat([_old, _row], ignore_index=True).drop_duplicates(subset=["dataset", "model", "seed"], keep="last")
_row.to_csv(STATUS_CSV, index=False)
print(f"Tersimpan ke: {STATUS_CSV}")


## 9. Breakdown Per-Kelas (WAJIB tiap run)


In [ ]:
import pandas as pd

model_final = YOLO(str(Path(SAVE_DIR) / "weights" / "best.pt"))
mf = model_final.val(data=STEEL_YAML, split="val", verbose=False)

print(f"=== {RUN_NAME} -- per-kelas mAP50:95 ===")
print(f"Agregat: mAP50={mf.box.map50:.4f}  mAP50:95={mf.box.map:.4f}  P={mf.box.mp:.4f}  R={mf.box.mr:.4f}\n")
try:
    _idx = [int(i) for i in mf.box.ap_class_index]
    AP50 = dict(zip(_idx, [float(v) for v in mf.box.ap50]))
    PREC = dict(zip(_idx, [float(v) for v in mf.box.p]))
    REC  = dict(zip(_idx, [float(v) for v in mf.box.r]))
except Exception as _e:
    print(f"(AP50/P/R per kelas tidak tersedia: {_e})"); AP50, PREC, REC = {}, {}, {}
# Bila checkpoint TIDAK menghasilkan satu pun deteksi benar (mis. run yang divergen sejak awal), Ultralytics tidak
# mengisi metrik per kelas (box.maps kosong) -> dicatat 0.0, sama dengan nilai yang akan dihitung framework.
if len(mf.box.ap_class_index) == 0:
    print("⚠️  Tidak ada deteksi benar sama sekali di val (mAP = 0) -- mAP per kelas dicatat 0.0.")
    MAPS = {ci: 0.0 for ci in mf.names}
else:
    MAPS = {ci: float(mf.box.maps[ci]) for ci in mf.names}
print(f"{'Kelas':16s} {'mAP50:95':>9s} {'AP50':>7s} {'P':>6s} {'R':>6s}")
for ci, cn in mf.names.items():
    print(f"{cn:16s} {MAPS[ci]:>9.4f} {AP50.get(ci, float('nan')):>7.4f} {PREC.get(ci, float('nan')):>6.3f} {REC.get(ci, float('nan')):>6.3f}")

# CSV BARU, terpisah dari CSV lama (per_class_results.csv / _P1FIX.csv). Kunci dedup = dataset+model+seed+class.
_ck = str(Path(SAVE_DIR) / "weights" / "best.pt")
rows = [{"dataset": CSV_TAG, "model": MODEL, "seed": SEED, "class": "ALL",
         "mAP50_95": mf.box.map, "mAP50": mf.box.map50, "precision": mf.box.mp, "recall": mf.box.mr,
         "status_latih": STATUS_LATIH, "deform_sha16": DEFORM_SHA256[:16], "ckpt_path": _ck}]
for ci, cn in mf.names.items():
    rows.append({"dataset": CSV_TAG, "model": MODEL, "seed": SEED, "class": cn,
                 "mAP50_95": MAPS[ci], "mAP50": AP50.get(ci), "precision": PREC.get(ci), "recall": REC.get(ci),
                 "status_latih": STATUS_LATIH, "deform_sha16": DEFORM_SHA256[:16], "ckpt_path": _ck})
df_new = pd.DataFrame(rows)
if Path(OUT_CSV).exists():
    df_out = pd.concat([pd.read_csv(OUT_CSV), df_new], ignore_index=True)
    df_out = df_out.drop_duplicates(subset=["dataset", "model", "seed", "class"], keep="last")
else:
    df_out = df_new
df_out.to_csv(OUT_CSV, index=False)
print(f"\nTersimpan ke: {OUT_CSV}  ({len(df_out)} baris total; {MODEL} seed{SEED}, {RUN_MODE})")
print("Kirim ke Claude setelah tiap run: CSV ini, status_latih_gridfix.csv, dan results.csv run ini.")


## Checklist per run

- [ ] Sel 0: torch = 2.11.0+cu128 (sesi baru: Run all → restart otomatis → Run all lagi)
- [ ] Sel 1: "Pengaman biaya aktif" tercetak
- [ ] Sel 2: GPU = A100 (dicek pertama), **tripwire `deform_ss2d.py` = GRID-FIX**, dataset steel-pipe COCOK (2.726 + 682 citra, 6.783 objek)
- [ ] Sel 3: MODEL / SEED / RUN_MODE benar; RUN_NAME berawalan `gf_`
- [ ] Sel 4a–4c: commit repo tercatat; patch NP-1…NP-4 + 3.1–3.3 + 6.1 OK; `selective_scan` ter-import; baris `Lingkungan run` dicatat
- [ ] Sel 5: pre-flight LOLOS — params PERSIS target, DeformSS2D sesuai, regresi P1 OK
- [ ] Sel 6: status checkpoint terbaca — run pertama tiap kombinasi HARUS "BARU"
- [ ] Sel 7–8: **150/150 epoch** (kalau terputus: Run all lagi → resume otomatis)
- [ ] Sel 8b: status latih tercatat (sehat / KOLAPS) — **run kolaps tidak diulang**
- [ ] Sel 9: per-kelas tercatat ke `per_class_results_gridfix.csv` (probe: `…_gridfix_probe.csv`)
- [ ] Centang baris di tabel progres (Sel 0); kirim CSV + `results.csv` ke Claude


## 10. Matikan Runtime (Wajib Dijalankan Terakhir)

In [ ]:
# Matikan runtime Colab otomatis setelah semua sel di atas selesai --
# supaya TIDAK terus memakan kuota Colab Pro kalau kamu tidak sempat
# matikan manual. Ini API resmi Google, benar-benar melepas VM-nya.
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)  # jeda singkat, jaga-jaga ada file yang masih proses ditulis ke Drive

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")
